# Name That LLM: TF-IDF + logistic regression

Generated by `python -m src.build_notebook` from
https://github.com/Dana-Lev/name-that-llm. The cells below recreate the `src/`
package, then fit on `train.csv` and write `submission.parquet`.

In [ ]:
from pathlib import Path

for empty_file in ['src/__init__.py', 'src/features/__init__.py', 'src/models/__init__.py']:
    Path(empty_file).parent.mkdir(parents=True, exist_ok=True)
    Path(empty_file).touch()

In [ ]:
%%writefile src/data.py
"""Loading the competition data and building the cross-validation folds."""

import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# The 18 valid labels, grouped by lab. Order is fixed: column k of every
# probability matrix in this project refers to LABELS[k].
LABS = {
    "OpenAI": ["gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra"],
    "Google": [
        "gemini-3.1-pro-preview",
        "gemini-3.6-flash",
        "gemini-3.7-flash",
        "gemini-3.8-flash",
    ],
    "Anthropic": [
        "claude-fable-5",
        "claude-opus-4-6",
        "claude-opus-4-8",
        "claude-opus-5",
        "claude-sonnet-5",
    ],
    "Qwen": ["qwen3.8-max"],
    "ZAI": ["glm-5.3", "glm-5.3-flash"],
    "Moonshot AI": ["kimi-k3"],
}
LABELS = [model for models in LABS.values() for model in models]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
LAB_OF = {model: lab for lab, models in LABS.items() for model in models}

N_SPLITS = 5
SEED = 42


def find_data_dir():
    """The folder with train.csv: $NTL_DATA_DIR, Kaggle's input folder, or ./data."""
    candidates = [
        os.environ.get("NTL_DATA_DIR"),
        "/kaggle/input/competitions/name-that-llm",
        "/kaggle/input/name-that-llm",
        "data",
    ]
    for candidate in candidates:
        if candidate and (Path(candidate) / "train.csv").exists():
            return Path(candidate)
    raise FileNotFoundError("train.csv not found; set NTL_DATA_DIR or put the data in ./data")


def load_train(data_dir="data"):
    """Read train.csv and add an integer `label` column."""
    df = pd.read_csv(Path(data_dir) / "train.csv")
    unknown = set(df["model"]) - set(LABELS)
    if unknown:
        raise ValueError(f"train.csv has labels outside the 18 valid ones: {sorted(unknown)}")
    df["label"] = df["model"].map(LABEL_TO_ID)
    return df


def load_test(data_dir="data"):
    return pd.read_csv(Path(data_dir) / "test.csv")


def make_folds(df, n_splits=N_SPLITS, seed=SEED):
    """Assign each row a fold id in [0, n_splits).

    All passages answering the same prompt land in the same fold, so every
    validation prompt is unseen during training, as in the hidden test set.
    Folds are also stratified by model to keep the class mix similar.
    """
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = np.full(len(df), -1, dtype=int)
    for fold, (_, valid_idx) in enumerate(
        splitter.split(df["text"], df["label"], groups=df["prompt_id"])
    ):
        folds[valid_idx] = fold
    return folds


In [ ]:
%%writefile src/features/tfidf.py
"""TF-IDF features over word and character n-grams."""

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion


def build_tfidf(word_ngrams=(1, 2), char_ngrams=(2, 5), min_df=2):
    """Word n-grams capture vocabulary; character n-grams capture punctuation
    and formatting habits. Case is kept because capitalisation is a style cue."""
    word = TfidfVectorizer(
        analyzer="word",
        ngram_range=word_ngrams,
        min_df=min_df,
        sublinear_tf=True,
        lowercase=False,
        token_pattern=r"(?u)\b\w+\b|[^\w\s]",
    )
    char = TfidfVectorizer(
        analyzer="char",
        ngram_range=char_ngrams,
        min_df=min_df,
        sublinear_tf=True,
        lowercase=False,
    )
    return FeatureUnion([("word", word), ("char", char)])


In [ ]:
%%writefile src/models/linear.py
"""Linear baseline: TF-IDF features into multinomial logistic regression."""

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from src.features.tfidf import build_tfidf


def build_tfidf_logreg(C=10.0, max_iter=2000, **tfidf_kwargs):
    """class_weight="balanced" matches the metric, which weights every model equally."""
    return Pipeline(
        [
            ("tfidf", build_tfidf(**tfidf_kwargs)),
            ("clf", LogisticRegression(C=C, max_iter=max_iter, class_weight="balanced")),
        ]
    )


In [ ]:
%%writefile src/predict.py
"""Fit the baseline on all of train.csv and write submission.parquet.

    python -m src.predict --data-dir data

This is what the Kaggle submission notebook runs.
"""

import argparse

import pandas as pd

from src.data import LABELS, find_data_dir, load_test, load_train
from src.models.linear import build_tfidf_logreg


def check_submission(submission, test):
    """Fail here rather than have Kaggle reject the file."""
    if list(submission.columns) != ["id", "model"]:
        raise ValueError(f"columns must be id, model; got {list(submission.columns)}")
    if submission.isna().any().any():
        raise ValueError("submission has null values")
    if submission["id"].duplicated().any() or set(submission["id"]) != set(test["id"]):
        raise ValueError("every test id must appear exactly once")
    unknown = set(submission["model"]) - set(LABELS)
    if unknown:
        raise ValueError(f"labels outside the 18 valid ones: {sorted(unknown)}")


def make_submission(data_dir=None, out="submission.parquet", build_model=build_tfidf_logreg):
    data_dir = find_data_dir() if data_dir is None else data_dir
    train, test = load_train(data_dir), load_test(data_dir)

    model = build_model()
    model.fit(train["text"], train["label"])
    predicted = model.predict(test["text"])

    submission = pd.DataFrame({"id": test["id"], "model": [LABELS[i] for i in predicted]})
    check_submission(submission, test)
    submission.to_parquet(out, index=False)
    print(f"trained on {len(train)} passages, wrote {len(submission)} predictions to {out}")
    return submission


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--data-dir", default=None)
    parser.add_argument("--out", default="submission.parquet")
    args = parser.parse_args()
    make_submission(args.data_dir, args.out)


In [ ]:
from src.predict import make_submission

submission = make_submission()
submission["model"].value_counts()